# Clamped beam with the common MORFE API

This conservative St. Venant–Kirchhoff example uses only the public, physics-independent MORFE workflow. It builds a reduced-order model of the first bending mode and reads the **backbone curve** off it: the amplitude-dependent oscillation frequency, which no linear spectrum can give.

Before the first run, run `julia setup.jl` in the bash from this repository. For more information look at the README.md of the repository.

`SVK` is only a short name for the structural backend used to describe the mechanical case; `build_model` and `parametrise` belong to the common API.

In [ ]:
using MORFE, MORFEFerrite
const SVK = MORFEFerrite.StructuralSVK # for Saint-Venant-Kirchhoff hyperelasticity

## 1. Describe the mechanical case

`order` is the polynomial expansion order. Order 9 is the reference calculation and costs about 35 seconds of solve; the whole notebook runs in under a minute. Because the cohomological solve is **graded**, a degree-`N` coefficient never depends on a higher degree, so orders 3, 5 and 7 are exact truncations of this one solve rather than three more runs.

`mechanical_model` reads the mesh and supplies the physical information required by the structural backend. Here the Rayleigh damping coefficients are zero, so the beam is conservative. `dirichlet = "Dirichlet"` names a physical facet group stored in the Gmsh mesh; every displacement component on those labeled facets is fixed to zero, producing the clamped ends. The finite-element and quadrature orders use their API defaults.

In [ ]:
order = 9
case = SVK.mechanical_model(joinpath(@__DIR__, "clamped_clamped_beam.msh");
    material = SVK.SVKMaterial(E = 160e3, ν = 0.22, ρ = 2.32e-3),
    damping = SVK.RayleighDamping(α = 0.0, β = 0.0),
    dirichlet = "Dirichlet")

## 2. Build the MORFE model

`build_model` converts the structural case into MORFE's physics-independent model and computes its spectral data. `master = [1]` selects the first vibration-mode pair as the tangent space of the reduced model. The returned `meta` contains auxiliary backend information, including the selected eigenvalues displayed below.

In [ ]:
(; model, spectral, meta) = build_model(case; master = [1], expansion_order = order)
meta.spectrum.eigenvalues[meta.master_indices] # print master eigenvalues

## 3. Parametrise the invariant manifold

`parametrise` computes the polynomial manifold map `W` and its reduced dynamics `R` up to the chosen order. The resonance configuration keeps near-resonant monomials in complex normal form. Evaluating `R` at the end of the cell displays the reduced system.

In [ ]:
W, R = parametrise(model, spectral, order;
    resonance = ResonanceConfig(style = :complex_normal_form, tol = 0.05))
R # print reduced dynamics

## 4. Read the backbone off R

In normal form the orbit is a circle $z_1 = \rho e^{i\Omega t}$. Every surviving monomial has $a - b = 1$, so substituting it gives every term of $R_1$ the same factor $e^{i\theta}$ and the phase drops out. Matching $\dot z_1 = (\dot\rho + i\rho\dot\theta)e^{i\theta}$ against it separates amplitude from phase:

$$\dot\rho = \mathrm{Re}\,R_1(\rho, \rho) \qquad\qquad \Omega = \frac{\mathrm{Im}\,R_1(\rho, \rho)}{\rho}$$

The beam is conservative, so $\mathrm{Re}\,R_1 \equiv 0$: every amplitude is a periodic orbit, and $\Omega$ is an explicit polynomial in $\rho$. That polynomial is the backbone, and `normal_form_branch` evaluates it.

$\rho$ is a coordinate on the manifold, not a length. `observable_polynomial` projects `W` onto one degree of freedom to recover a physical displacement, and `cycle_amplitude` reads its half peak-to-peak excursion over one cycle. Node 289 sits at mid-span on the top surface, where the first bending mode peaks.

In [ ]:
u = observable_polynomial(W, SVK.probe_dof(case, 289, 2)) # y displacement at mid-span

thickness = 10.0                  # beam thickness, mesh length units
ρ = range(0, 85; length = 341)    # modal amplitude; ρ = 85 is ≈ 1.1 × thickness at the probe
ω₀ = normal_form_branch(R; amplitudes = [0.0]).frequency[1]

curves = map(3:2:order) do N
    b = normal_form_branch(restrict_ReducedDynamics_to_degree(R, N); amplitudes = ρ)
    P = MORFE.Polynomials.restrict_polynomial_to_degree(u, N)
    (; N, b.amplitude, b.frequency, dω = 100 .* (b.frequency ./ ω₀ .- 1),
        a = cycle_amplitude.(Ref(P), b.amplitude) ./ thickness)
end

## 5. Plot the backbone

The beam stiffens as it swings: at a displacement of about three quarters of its thickness the frequency has risen roughly 13 % above the linear eigenfrequency. Plotting the four truncations together shows where the expansion stops converging, which is the honest way to read the useful range of a ROM. They agree near the origin and separate as the amplitude grows.

In [ ]:
using CairoMakie
fig = Figure(size = (520, 380))
ax = Axis(fig[1, 1]; xlabel = "Δω / ω₀  [%]", ylabel = "displacement / thickness  [%]")
foreach(c -> lines!(ax, c.dω, 100 .* c.a; label = "order $(c.N)"), curves)
axislegend(ax; position = :lt)
fig

## 6. Save the ROM and the backbone

The common saver writes `W`, `R`, their coefficient table and a summary to the example's `results` directory. `drop_below = 0.0` keeps every coefficient: the degree-9 resonant term is of order `1e-17`, and the default cut of `1e-14` would discard the very row the reference is blessed on.

The two CSVs alongside it are the backbone this notebook computed and the probe row of `W` it was computed from. The first is the data the MORFE website's backbone figure draws; the second is what makes that figure reproducible, since `W.jls` is not committed.

In [ ]:
MORFE.save_rom(joinpath(@__DIR__, "results"), W, R; drop_below = 0.0)

open(joinpath(@__DIR__, "results", "data", "backbone.csv"), "w") do io
    println(io, "order,r,omega,omega_ratio,amplitude,amplitude_ratio")
    for c in curves, i in eachindex(c.amplitude)
        println(io, join((c.N, c.amplitude[i], c.frequency[i], c.frequency[i] / ω₀,
            thickness * c.a[i], c.a[i]), ","))
    end
end

open(joinpath(@__DIR__, "results", "data", "W_probe_coefficients.csv"), "w") do io
    println(io, "exp_1,exp_2,W_y_re,W_y_im")
    for (e, c) in zip(MORFE.Polynomials.multiindex_set(u).exponents,
        MORFE.Polynomials.coefficients(u))
        println(io, join((e[1], e[2], real(c), imag(c)), ","))
    end
end